# Layer 1: Metric Shift — Transformer + KG (Retrained)

Same as Model 2's Layer 1 but for the KG-augmented transformer. The KG regularization
pulls connected books together during training, which should make item embeddings more
stable — we test whether that shows up as smaller item centroid shift.

- **Item centroid shift** (≈ bi shift): how far did each genre's item-embedding centroid move from ORIGINAL?
- **User centroid shift** (≈ bu shift): how far did each genre's user-embedding centroid move?

Output: `layer1/result/centroid_shift.csv` + figures.

In [ ]:
import pickle, ast, re
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

BASE = Path('/home/moshtasa/Research/phd-recsys/Book/current/PerturbRec')
ORIGINAL_CSV = BASE / 'Data' / 'ORIGINAL_books.csv'
TDIR = BASE / 'transformer_kg'
EMB_DIR = TDIR / 'result' / 'embeddings'          # KG-augmented retrained embeddings

USER_COL, BOOK_COL, GENRE_COL = 'user_id', 'book_id', 'genres'
GENRES = ['Drama']
N_FILTER = {2, 4, 6, 25, 50, 100, 200, 500, 1000, 2000}
PATTERN = re.compile(r'^f_(.+)_(\d+)u_pos(\d+)_neg(NA|0|1|\d+)_.+\.csv$')

print('Config loaded. Embeddings:', EMB_DIR)


In [ ]:
## Shared utilities

def parse_genres(s):
    if pd.isna(s): return []
    s = str(s).strip()
    if not s: return []
    if (s.startswith('[') and s.endswith(']')) or (s.startswith('(') and s.endswith(')')):
        try:
            p = ast.literal_eval(s)
            if isinstance(p, (list, tuple)):
                return [str(x).strip().strip('"').strip("'") for x in p if str(x).strip()]
        except: pass
    for sep in [',', '|', ';', '//', '/']:
        if sep in s:
            return [t.strip().strip('"').strip("'") for t in s.split(sep) if t.strip()]
    return [s.strip().strip('"').strip("'")]

def book_genres_map():
    df = pd.read_csv(ORIGINAL_CSV, usecols=[BOOK_COL, GENRE_COL], low_memory=False)
    df = df.drop_duplicates(subset=[BOOK_COL])
    df[GENRE_COL] = df[GENRE_COL].fillna('').astype(str)
    return {int(r[BOOK_COL]): parse_genres(r[GENRE_COL]) for _, r in df.iterrows()}

def user_dominant_genre_map(bg):
    df = pd.read_csv(ORIGINAL_CSV, usecols=[USER_COL, BOOK_COL], low_memory=False)
    df[USER_COL] = df[USER_COL].astype(int)
    df[BOOK_COL] = df[BOOK_COL].astype(int)
    out = {}
    for uid, g in df.groupby(USER_COL):
        gs = []
        for b in g[BOOK_COL]: gs.extend(bg.get(int(b), []))
        out[int(uid)] = Counter(gs).most_common(1)[0][0] if gs else 'Unknown'
    return out

def load_art(path):
    with open(path, 'rb') as f: return pickle.load(f)

def scan_injections(emb_dir):
    jobs = []
    for fp in sorted(emb_dir.glob('f_*_model.pt')):
        stem = fp.stem.replace('_model', '')
        m = PATTERN.match(stem + '.csv')
        if m and int(m.group(3)) == 5 and int(m.group(2)) in N_FILTER and m.group(1) in GENRES:
            jobs.append((fp, stem, m.group(1), int(m.group(2))))
    return sorted(jobs, key=lambda x: (x[2], x[3]))

bg = book_genres_map()
ulabels = user_dominant_genre_map(bg)
orig_uids_set = set(ulabels.keys())
print(f'Loaded {len(ulabels)} original user labels')


In [ ]:
OUT = TDIR / 'layer1' / 'result'
(OUT / 'figure').mkdir(parents=True, exist_ok=True)

def compute_centroid_shift():
    art0 = load_art(EMB_DIR / 'ORIGINAL_model.pt')
    qi0, iid0 = art0['qi'].astype(np.float32), art0['inner_iid_to_raw']
    pu0, uid0 = art0['pu'].astype(np.float32), art0['inner_uid_to_raw']

    item_c0, user_c0 = {}, {}
    for g in GENRES + ['All']:
        if g == 'All':
            item_c0[g] = qi0.mean(axis=0); user_c0[g] = pu0.mean(axis=0)
        else:
            imask = [g in bg.get(int(i), []) for i in iid0]
            if any(imask): item_c0[g] = qi0[imask].mean(axis=0)
            ulabs = [ulabels.get(int(u), 'Unknown') for u in uid0]
            umask = [l == g for l in ulabs]
            if any(umask): user_c0[g] = pu0[umask].mean(axis=0)

    results = []
    for fp, name, genre, n in scan_injections(EMB_DIR):
        art = load_art(fp)
        qi, iid = art['qi'].astype(np.float32), art['inner_iid_to_raw']
        pu, uid = art['pu'].astype(np.float32), art['inner_uid_to_raw']
        for g in GENRES + ['All']:
            if g == 'All':
                ci = qi.mean(axis=0)
            else:
                imask = [g in bg.get(int(i), []) for i in iid]
                if not any(imask): continue
                ci = qi[imask].mean(axis=0)
            if g in item_c0:
                results.append({'base': name, 'genre': g, 'n': n, 'metric': 'item_centroid_shift', 'shift': float(np.linalg.norm(ci - item_c0[g]))})
        for g in GENRES + ['All']:
            if g == 'All':
                cu = pu.mean(axis=0)
            else:
                ulabs = [ulabels.get(int(u), genre) for u in uid]
                umask = [l == g for l in ulabs]
                if not any(umask): continue
                cu = pu[umask].mean(axis=0)
            if g in user_c0:
                results.append({'base': name, 'genre': g, 'n': n, 'metric': 'user_centroid_shift', 'shift': float(np.linalg.norm(cu - user_c0[g]))})
        print(f'  {name}', flush=True)

    df = pd.DataFrame(results)
    df.to_csv(OUT / 'centroid_shift.csv', index=False)
    return df

df = compute_centroid_shift()
print(f'Layer 1: {len(df)} rows')


In [ ]:
## Plot centroid shift vs n

colors = {'Drama': 'steelblue', 'All': 'gray'}
for metric, title in [('item_centroid_shift', 'Item'), ('user_centroid_shift', 'User')]:
    fig, ax = plt.subplots(figsize=(8, 6))
    sub = df[df['metric'] == metric].sort_values(['genre', 'n'])
    for g, s in sub.groupby('genre'):
        ax.plot(s['n'], s['shift'], 'o-', c=colors.get(g, 'gray'), label=g)
    ax.set_xscale('log'); ax.set_xlabel('Injected biasing users (n)')
    ax.set_ylabel(f'{title} centroid shift (L2 from ORIGINAL)')
    ax.set_title(f'Layer 1 [KG]: {title} centroid shift vs n')
    ax.legend(); fig.tight_layout()
    fig.savefig(OUT / 'figure' / f'{metric}_vs_n.png', dpi=150)
    plt.close(fig)
    print(f'  saved -> {metric}_vs_n.png')

print('Layer 1 complete.')
